In [ ]:
import torch
import numpy as np
import custom_gym
import matplotlib
# matplotlib.use('Agg')
%matplotlib inline

from  dynamic_systems_rl import SinglePendulum


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

dt=0.01;

state_max = torch.tensor([1.,1., 10.]) # theta, dtheta
state_min = -state_max
mass = 1.0; length=1.0; g= 9.81; coef_viscous = 0.001
action_max = torch.tensor([0.5*mass*g*length])
w_goal=1
w_action=1
agent = SinglePendulum(state_min=state_min, state_max=state_max,
                       mass=mass, length=length, coef_viscous=coef_viscous,
                       w_goal=w_goal, w_action=w_action,
                       action_max=action_max, use_gym=False)


In [ ]:
reset_bound = np.array([1.,1.0, 0.1]) # theta, dtheta
env = custom_gym.CreateGymEnv(agent, 
                              reset_bound=reset_bound,     
                              dt=dt)

In [ ]:
from stable_baselines3.common.env_checker import check_env
# It will check your custom environment and output additional warnings if needed
check_env(env)

In [ ]:
from stable_baselines3 import PPO, SAC, DDPG
# # Instantiate the agent
policy_kwargs = dict(net_arch=[64, 64])

model = SAC(
    "MlpPolicy",
    env,
    gamma=0.99,
    use_sde=True,
    sde_sample_freq=4,
    learning_rate=1e-3,
    verbose=1,
    policy_kwargs=policy_kwargs
)



In [ ]:
def policy(state):
    action, _ = model.predict(state, deterministic=True)
    return action

def callback(agent, policy, dt, animation=True, file_name='pendu'):
    print("Testing....")
    state = torch.tensor([[0.,1.,0.]]).view(1,3)
    history = []
    
    T=int(10/dt)
    traj = torch.zeros(T+1,3)
    cum_reward = torch.tensor([0.]*state.shape[0])
    action_history = torch.zeros(state.shape[0],T,1) #batch_size x T x 1
    for i in range(T):
        action = torch.from_numpy(policy(state)) # batch_size x 1
        action_history[:,i,:] = action
        r = agent.reward_state_action(state,action)#reward_test(state,action)
        cum_reward+=r #reward_test(state,action)
        state = agent.forward_simulate(state,action,dt)
        traj[i+1,:] = state.clone().cpu()
    theta_t = torch.arctan2(traj[:,0],traj[:,1]).cpu()
    dtheta_t = traj[:,1].cpu()
    from plot_utils import plt_pendulum
    plt=plt_pendulum(theta_t.to('cpu').numpy(), 
                    figsize=5, dt=dt, scale=10, skip=1, animation=animation)
    plt.show()
    total_reward = torch.mean(cum_reward)
    return r.mean().to('cpu'), total_reward.to('cpu')
    

In [ ]:
import time
T = 0
for i in range(10):
    t1 = time.time()
    model.learn(total_timesteps=int(1e5))
    t2 = time.time()
    T = T+ (t2-t1)
    print("Time Taken: ", t2-t1)
    print("Total time: ", T)
    model.save('pendulum')
    r_m, r_cum = callback(agent=agent,policy=policy, dt=dt)
    print("mean, cum reward, mu: ", r_m, r_cum)